In [1]:
# 1. Install required packages silently
!pip install -q -U bitsandbytes accelerate transformers rouge-score

# 2. Force Python to recognize the newly installed packages without a restart
import site
import importlib
importlib.reload(site)

# 3. Now it is safe to import your heavy libraries
print("Environment prepped. Safe to load bitsandbytes.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 90.4 MB/s eta 0:00:00
Environment prepped. Safe to load bitsandbytes.


In [2]:
import os, gc, time, json, math, warnings, traceback
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GRPC_VERBOSITY"] = "ERROR"
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
from rouge_score import rouge_scorer as rs_lib

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache(); gc.collect()

# ════════════════════════════════════════════════════════════
# PATHS & CONSTANTS
# ════════════════════════════════════════════════════════════
TARGET_ID          = "Qwen/Qwen2.5-7B-Instruct"
LONGBENCH_DATA_DIR = "/kaggle/input/datasets/mihirkhatri1710/longbench/data"
LOG_DIR            = "/kaggle/working/lorc_sweep"
os.makedirs(LOG_DIR, exist_ok=True)

HEAD_DIM = 128
N_LAYERS = 28
# All english single-doc + multi-doc subsets in LongBench
ALL_SUBSETS = [
    "multifieldqa_en",
    "narrativeqa",
    "qasper",
    "hotpotqa",
    "2wikimqa",
    "musique",
    "gov_report",
    "qmsum",
    "multi_news",
    "trec",
    "triviaqa",
    "samsum",
    "passage_count",
    "passage_retrieval_en",
    "lcc",
    "repobench-p",
]

# ════════════════════════════════════════════════════════════
# HYPERPARAMETER GRID  — ordered from LEAST to MOST memory
# Logic: smaller context + bigger window + lower rank = less memory used
# We escalate through the grid until OOM
# ════════════════════════════════════════════════════════════
PARAM_GRID = [
    # (context_truncation, recent_window, energy_thresh, rank_early, rank_mid, rank_late)
    # Stage 1 — minimum memory: short context, large window, conservative ranks
    (1024,  256, 0.95, 96,  80,  64),
    (1024,  128, 0.92, 96,  80,  64),
    (2000,  256, 0.92, 96,  80,  64),
    (2000,  128, 0.90, 112, 96,  80),
    # Stage 2 — moderate memory
    (2000,   64, 0.88, 112, 96,  80),
    (3000,  128, 0.90, 112, 96,  80),
    (3000,   64, 0.88, 112, 96,  80),
    # Stage 3 — higher memory (original settings)
    (4000,  128, 0.92, 112, 96,  80),
    (4000,   64, 0.88, 112, 96,  80),
    # Stage 4 — maximum compression (lowest ranks)
    (4000,   64, 0.85, 96,  80,  64),
    (4000,   32, 0.85, 96,  80,  64),
    (4000,   32, 0.80, 80,  64,  48),
]

# ════════════════════════════════════════════════════════════
# MODEL  — load once, reuse across entire sweep
# ════════════════════════════════════════════════════════════
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_ID, quantization_config=bnb_config,
    device_map="auto", attn_implementation="eager"
)
target_model.eval()
_rouge = rs_lib.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
print("Model loaded.\n")

# ════════════════════════════════════════════════════════════
# ROPE UTILS
# ════════════════════════════════════════════════════════════

def build_rope_cache(seq_len, head_dim=HEAD_DIM, base=1_000_000, device="cpu"):
    half     = head_dim // 2
    inv_freq = 1.0 / (base ** (torch.arange(0, half, dtype=torch.float32) / half))
    positions = torch.arange(seq_len, dtype=torch.float32)
    angles    = torch.outer(positions, inv_freq)
    angles    = torch.cat([angles, angles], dim=-1)
    return angles.cos().to(device), angles.sin().to(device)

def rotate_half(x):
    half = x.shape[-1] // 2
    return torch.cat([-x[..., half:], x[..., :half]], dim=-1)

def apply_rope(x, cos, sin, offset=0):
    s  = x.shape[2]
    c  = cos[offset:offset+s].unsqueeze(0).unsqueeze(0)
    si = sin[offset:offset+s].unsqueeze(0).unsqueeze(0)
    return (x * c) + (rotate_half(x) * si)

def undo_rope(x, cos, sin, offset=0):
    s  = x.shape[2]
    c  = cos[offset:offset+s].unsqueeze(0).unsqueeze(0)
    si = sin[offset:offset+s].unsqueeze(0).unsqueeze(0)
    return (x * c) - (rotate_half(x) * si)

# ════════════════════════════════════════════════════════════
# CACHE UTILS
# ════════════════════════════════════════════════════════════


def to_tuple(cache):
    """
    Normalize any HF cache → tuple of (k, v)
    Handles:
    - DynamicCache
    - tuple/list with extra fields (k, v, ...)
    """
    if hasattr(cache, "key_cache"):
        return tuple((k, v) for k, v in zip(cache.key_cache, cache.value_cache))

    out = []
    for layer in cache:
        if isinstance(layer, (list, tuple)):
            k = layer[0]
            v = layer[1]
            out.append((k, v))
        else:
            raise ValueError(f"Unknown cache format: {type(layer)}")

    return tuple(out)

def to_dynamic(layers):
    """
    Safely rebuild DynamicCache
    """
    c = DynamicCache()
    for i, layer in enumerate(layers):
        k, v = layer[0], layer[1]
        c.update(k, v, layer_idx=i)
    return c

# ════════════════════════════════════════════════════════════
# SVD CORE
# ════════════════════════════════════════════════════════════

def layer_rank(idx, n, rank_early, rank_mid, rank_late):
    f = idx / max(n - 1, 1)
    if f < 0.32: return rank_early
    if f < 0.68: return rank_mid
    return rank_late

def svd_compress(tensor, rank, energy_thresh):
    b, h, s, d = tensor.shape
    rank = min(rank, d, s)
    mat  = tensor.float().reshape(b * h, s, d)
    try:
        U, S, Vh = torch.linalg.svd(mat, full_matrices=False)
        cap   = (S[:, :rank]**2).sum(-1, keepdim=True)
        total = (S**2).sum(-1, keepdim=True)
        mask  = (cap / (total + 1e-9) >= energy_thresh).unsqueeze(-1).expand_as(mat)
        approx = (U[:, :, :rank] * S[:, :rank].unsqueeze(1)) @ Vh[:, :rank, :]
        result = torch.where(mask, approx, mat)
    except Exception:
        return tensor
    return result.reshape(b, h, s, d).to(tensor.dtype)

# ════════════════════════════════════════════════════════════
# COMPRESSION FUNCTIONS  (parametrised — no globals)
# ════════════════════════════════════════════════════════════

def compress_rope_aware(cache, window, energy_thresh, rank_early, rank_mid, rank_late):
    layers   = to_tuple(cache)
    n        = len(layers)
    seq_len  = layers[0][0].shape[2]
    device   = layers[0][0].device
    cos, sin = build_rope_cache(seq_len + 64, HEAD_DIM, 1_000_000, device)
    new = []
    for i, (k, v) in enumerate(layers):
        s = k.shape[2]
        if s <= window:
            new.append((k, v)); continue
        r      = layer_rank(i, n, rank_early, rank_mid, rank_late)
        hk, rk = k[:, :, :-window, :], k[:, :, -window:, :]
        hv, rv = v[:, :, :-window, :], v[:, :, -window:, :]
        # Keys: undo RoPE → compress → reapply
        pre    = undo_rope(hk.float(), cos, sin, 0).to(hk.dtype)
        cpr    = svd_compress(pre, r, energy_thresh)
        post   = apply_rope(cpr.float(), cos, sin, 0).to(hk.dtype)
        # Values: compress directly
        new.append((torch.cat([post, rk], 2),
                    torch.cat([svd_compress(hv, r, energy_thresh), rv], 2)))
    return to_dynamic(tuple(new))

def compress_standard(cache, window, energy_thresh, rank_early, rank_mid, rank_late):
    layers = to_tuple(cache)
    n      = len(layers)
    new    = []
    for i, (k, v) in enumerate(layers):
        if k.shape[2] <= window:
            new.append((k, v)); continue
        r      = layer_rank(i, n, rank_early, rank_mid, rank_late)
        hk, rk = k[:, :, :-window, :], k[:, :, -window:, :]
        hv, rv = v[:, :, :-window, :], v[:, :, -window:, :]
        new.append((torch.cat([svd_compress(hk, r, energy_thresh), rk], 2),
                    torch.cat([svd_compress(hv, r, energy_thresh), rv], 2)))
    return to_dynamic(tuple(new))

def mem_saved_mb(cache, window, rank_early, rank_mid, rank_late):
    tc = to_tuple(cache)
    if not tc: return 0.0
    b, h, s, d = tc[0][0].shape
    if s <= window: return 0.0
    hist  = s - window
    total = 0
    for i in range(len(tc)):
        r     = min(layer_rank(i, len(tc), rank_early, rank_mid, rank_late), d, hist)
        saved = max(hist * d - hist * r, 0)
        total += saved * h * 2 * 2
    return total / (1024 ** 2)

# ════════════════════════════════════════════════════════════
# PROMPT FORMATTING
# ════════════════════════════════════════════════════════════

def format_prompt(ctx, q, max_ctx_tokens):
    system   = "You are a helpful assistant. Answer concisely based only on the passage."
    head     = f"<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\nPassage:\n"
    tail     = f"\n\nQuestion: {q}<|im_end|>\n<|im_start|>assistant\n"
    head_ids = tokenizer.encode(head, add_special_tokens=False)
    tail_ids = tokenizer.encode(tail, add_special_tokens=False)
    budget   = max_ctx_tokens - len(head_ids) - len(tail_ids) - 4
    ctx_ids  = tokenizer.encode(ctx, add_special_tokens=False)
    if len(ctx_ids) > budget:
        f = int(budget * 0.6)
        ctx_ids = ctx_ids[:f] + ctx_ids[-(budget - f):]
    ids = head_ids + ctx_ids + tail_ids
    t   = torch.tensor([ids])
    return {"input_ids": t, "attention_mask": torch.ones_like(t)}

# ════════════════════════════════════════════════════════════
# GENERATION
# ════════════════════════════════════════════════════════════

def run_baseline(tokens, max_new=128):
    inp  = {k: v.to(target_model.device) for k, v in tokens.items()}
    plen = inp["input_ids"].shape[-1]
    with torch.no_grad():
        out = target_model.generate(
            **inp, max_new_tokens=max_new, do_sample=False,
            temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0, plen:], skip_special_tokens=True).strip()

def run_lorc(tokens, compress_fn, window, max_new=128):
    inp  = {k: v.to(target_model.device) for k, v in tokens.items()}
    plen = inp["input_ids"].shape[-1]
    gen  = inp["input_ids"].clone()
    with torch.no_grad():
        out = target_model(**inp, use_cache=True)
    cache = compress_fn(out.past_key_values)
    for step in range(max_new):
        with torch.no_grad():
            out = target_model(gen[:, -1:], past_key_values=cache, use_cache=True)
        logits = torch.nan_to_num(out.logits[:, -1, :], nan=-1e9, posinf=1e9, neginf=-1e9)
        cache  = out.past_key_values
        if (step + 1) % 128 == 0:
            cache = compress_fn(cache)
        tok = torch.argmax(logits, -1)
        gen = torch.cat([gen, tok.view(1, 1)], -1)
        if tok.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(gen[0, plen:], skip_special_tokens=True).strip()

def compute_ppl(text):
    if not text or len(text.strip()) < 3: return float('inf')
    enc = tokenizer(text, return_tensors="pt", truncation=True,
                    max_length=150, add_special_tokens=False)
    ids = enc.input_ids.to(target_model.device)
    if ids.shape[1] < 2: return float('inf')
    with torch.no_grad():
        loss = target_model(ids, labels=ids).loss.item()
    if math.isnan(loss) or math.isinf(loss) or loss > 15: return float('inf')
    return round(math.exp(loss), 3)

# ════════════════════════════════════════════════════════════
# DATA LOADER
# ════════════════════════════════════════════════════════════

def load_subset(subset, data_dir=LONGBENCH_DATA_DIR):
    path = os.path.join(data_dir, f"{subset}.jsonl")
    if not os.path.exists(path):
        return None
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            if isinstance(row.get("answers"), str):
                row["answers"] = json.loads(row["answers"])
            rows.append(row)
    return rows

# ════════════════════════════════════════════════════════════
# LOGGING UTILS  — write every sample immediately so no data lost on OOM
# ════════════════════════════════════════════════════════════

def log_line(fpath, line):
    """Append one line to a text log file, flushed immediately."""
    with open(fpath, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def log_sample(fpath, record):
    """Write a JSON record for one sample, one line, immediately flushed."""
    with open(fpath, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def vram_used_mb():
    """Current VRAM usage across all GPUs in MB."""
    total = 0
    for i in range(torch.cuda.device_count()):
        total += torch.cuda.memory_allocated(i)
    return total / (1024 ** 2)

# ════════════════════════════════════════════════════════════
# SINGLE PARAM CONFIG RUN
# ════════════════════════════════════════════════════════════

def run_one_config(config_idx, ctx_len, window, energy, r_early, r_mid, r_late,
                   subsets_to_run, max_per_subset=150):
    """
    Run one hyperparameter configuration across all requested subsets.
    Returns True if completed, False if hit OOM.
    Writes logs immediately per sample — crash-safe.
    """
    cfg_tag  = f"cfg{config_idx:02d}_ctx{ctx_len}_w{window}_e{int(energy*100)}_r{r_early}-{r_mid}-{r_late}"
    log_txt  = os.path.join(LOG_DIR, f"{cfg_tag}.txt")
    log_jsonl = os.path.join(LOG_DIR, f"{cfg_tag}.jsonl")

    # Partial compress fns bound to this config
    def cfn_rope(cache):
        return compress_rope_aware(cache, window, energy, r_early, r_mid, r_late)
    def cfn_std(cache):
        return compress_standard(cache, window, energy, r_early, r_mid, r_late)

    header = (f"=== Config {config_idx} | ctx={ctx_len} window={window} "
              f"energy={energy} ranks={r_early}/{r_mid}/{r_late} ===")
    print(header)
    log_line(log_txt, header)
    log_line(log_txt, f"Started: {time.strftime('%Y-%m-%d %H:%M:%S')}")

    total_base_rL, total_std_rL, total_rope_rL = [], [], []
    total_mem_std, total_mem_rope = [], []

    for subset in subsets_to_run:
        rows = load_subset(subset)
        if rows is None:
            msg = f"  [SKIP] {subset} — file not found"
            print(msg); log_line(log_txt, msg)
            continue

        rows = rows[:max_per_subset]
        sub_header = f"\n--- Subset: {subset} ({len(rows)} samples) ---"
        print(sub_header); log_line(log_txt, sub_header)

        base_rL, std_rL, rope_rL = [], [], []
        mem_std_list, mem_rope_list = [], []

        for i, row in enumerate(rows):
            ctx  = row.get("context", "")
            q    = row.get("input", "")
            gold = (row.get("answers") or [""])[0]

            tokens = format_prompt(ctx, q, ctx_len)
            sample_log = {
                "config": cfg_tag, "subset": subset, "sample": i,
                "question": q[:80], "gold": gold[:80]
            }

            # ── Baseline ──────────────────────────────────
            try:
                t0    = time.time()
                out_b = run_baseline(tokens)
                t_b   = round(time.time() - t0, 2)
                sc_b  = _rouge.score(gold, out_b)
                ppl_b = compute_ppl(out_b)
                rL_b  = round(sc_b["rougeL"].fmeasure, 4)
                base_rL.append(rL_b)
                sample_log["baseline"] = {"rL": rL_b, "ppl": ppl_b, "t": t_b, "out": out_b[:150]}
                torch.cuda.empty_cache()
            except torch.cuda.OutOfMemoryError:
                msg = f"  OOM at baseline | subset={subset} sample={i} config={cfg_tag}"
                print(msg); log_line(log_txt, msg)
                log_sample(log_jsonl, {**sample_log, "oom": "baseline"})
                return False

            # ── Standard LoRC ──────────────────────────────
            try:
                t0    = time.time()
                out_s = run_lorc(tokens, cfn_std, window)
                t_s   = round(time.time() - t0, 2)
                sc_s  = _rouge.score(gold, out_s)
                ppl_s = compute_ppl(out_s)
                rL_s  = round(sc_s["rougeL"].fmeasure, 4)
                mem_s = round(mem_saved_mb(
                    compress_standard(
                        __build_prefill_cache(tokens),
                        window, energy, r_early, r_mid, r_late
                    ), window, r_early, r_mid, r_late
                ), 2) if False else 0.0  # skip extra prefill cost, use approx
                std_rL.append(rL_s)
                sample_log["lorc_std"] = {"rL": rL_s, "ppl": ppl_s, "t": t_s, "out": out_s[:150]}
                torch.cuda.empty_cache()
            except torch.cuda.OutOfMemoryError:
                msg = f"  OOM at std_lorc | subset={subset} sample={i} config={cfg_tag}"
                print(msg); log_line(log_txt, msg)
                log_sample(log_jsonl, {**sample_log, "oom": "std_lorc"})
                return False

            # ── RoPE-Aware LoRC ────────────────────────────
            try:
                t0    = time.time()
                out_r = run_lorc(tokens, cfn_rope, window)
                t_r   = round(time.time() - t0, 2)
                sc_r  = _rouge.score(gold, out_r)
                ppl_r = compute_ppl(out_r)
                rL_r  = round(sc_r["rougeL"].fmeasure, 4)
                rope_rL.append(rL_r)
                sample_log["lorc_rope"] = {"rL": rL_r, "ppl": ppl_r, "t": t_r, "out": out_r[:150]}
                torch.cuda.empty_cache()
            except torch.cuda.OutOfMemoryError:
                msg = f"  OOM at rope_lorc | subset={subset} sample={i} config={cfg_tag}"
                print(msg); log_line(log_txt, msg)
                log_sample(log_jsonl, {**sample_log, "oom": "rope_lorc"})
                return False

            # ── Per-sample memory estimate ──────────────────
            # Cheap approximation: just compute from cache shape, no extra prefill
            try:
                inp_device = {k: v.to(target_model.device) for k, v in tokens.items()}
                with torch.no_grad():
                    tmp_out = target_model(**inp_device, use_cache=True)
                c_std  = compress_standard(tmp_out.past_key_values, window, energy, r_early, r_mid, r_late)
                c_rope = compress_rope_aware(tmp_out.past_key_values, window, energy, r_early, r_mid, r_late)
                mem_s_mb = round(mem_saved_mb(c_std,  window, r_early, r_mid, r_late), 2)
                mem_r_mb = round(mem_saved_mb(c_rope, window, r_early, r_mid, r_late), 2)
                del tmp_out, c_std, c_rope
                torch.cuda.empty_cache()
                mem_std_list.append(mem_s_mb)
                mem_rope_list.append(mem_r_mb)
                sample_log["mem_std_mb"]  = mem_s_mb
                sample_log["mem_rope_mb"] = mem_r_mb
            except Exception:
                mem_s_mb = mem_r_mb = 0.0

            # Print + log immediately
            vram = round(vram_used_mb(), 1)
            line = (f"  [{i+1:>3}/{len(rows)}] B:{rL_b:.3f} "
                    f"Std:{rL_s:.3f} RoPE:{rL_r:.3f} "
                    f"MemStd:{mem_s_mb:.1f}MB MemRoPE:{mem_r_mb:.1f}MB "
                    f"VRAM:{vram:.0f}MB")
            print(line)
            log_line(log_txt, line)
            log_sample(log_jsonl, sample_log)

        # Subset summary
        def m(lst): return round(float(np.mean(lst)), 4) if lst else 0.0
        sub_summary = (
            f"  >> {subset} SUMMARY: "
            f"Base={m(base_rL)} Std={m(std_rL)} RoPE={m(rope_rL)} "
            f"AvgMemStd={m(mem_std_list):.1f}MB AvgMemRoPE={m(mem_rope_list):.1f}MB "
            f"RoPE_wins={sum(1 for a,b in zip(rope_rL,base_rL) if a>b)}/"
            f"ties={sum(1 for a,b in zip(rope_rL,base_rL) if a==b)}/"
            f"losses={sum(1 for a,b in zip(rope_rL,base_rL) if a<b)}"
        )
        print(sub_summary); log_line(log_txt, sub_summary)
        total_base_rL += base_rL
        total_std_rL  += std_rL
        total_rope_rL += rope_rL
        total_mem_std += mem_std_list
        total_mem_rope += mem_rope_list

    # Config-level summary
    def m(lst): return round(float(np.mean(lst)), 4) if lst else 0.0
    cfg_summary = (
        f"\n=== CONFIG {config_idx} FINAL SUMMARY ===\n"
        f"  Baseline  ROUGE-L : {m(total_base_rL)}\n"
        f"  Std LoRC  ROUGE-L : {m(total_std_rL)}  (delta={m(total_std_rL)-m(total_base_rL):+.4f})\n"
        f"  RoPE LoRC ROUGE-L : {m(total_rope_rL)}  (delta={m(total_rope_rL)-m(total_base_rL):+.4f})\n"
        f"  RoPE vs Std delta : {m(total_rope_rL)-m(total_std_rL):+.4f}\n"
        f"  Avg MemStd  : {m(total_mem_std):.2f} MB\n"
        f"  Avg MemRoPE : {m(total_mem_rope):.2f} MB\n"
        f"  RoPE W/T/L vs Base: "
        f"{sum(1 for a,b in zip(total_rope_rL,total_base_rL) if a>b)}/"
        f"{sum(1 for a,b in zip(total_rope_rL,total_base_rL) if a==b)}/"
        f"{sum(1 for a,b in zip(total_rope_rL,total_base_rL) if a<b)}\n"
        f"  Std  W/T/L vs Base: "
        f"{sum(1 for a,b in zip(total_std_rL,total_base_rL) if a>b)}/"
        f"{sum(1 for a,b in zip(total_std_rL,total_base_rL) if a==b)}/"
        f"{sum(1 for a,b in zip(total_std_rL,total_base_rL) if a<b)}\n"
        f"  Total samples evaluated: {len(total_base_rL)}\n"
        f"  Completed: {time.strftime('%Y-%m-%d %H:%M:%S')}"
    )
    print(cfg_summary); log_line(log_txt, cfg_summary)

    # Save aggregated JSON for this config
    agg_path = os.path.join(LOG_DIR, f"{cfg_tag}_summary.json")
    with open(agg_path, "w") as f:
        json.dump({
            "config": {
                "ctx_len": ctx_len, "window": window, "energy": energy,
                "ranks": {"early": r_early, "mid": r_mid, "late": r_late}
            },
            "n_samples": len(total_base_rL),
            "baseline_rL":  m(total_base_rL),
            "std_rL":       m(total_std_rL),
            "rope_rL":      m(total_rope_rL),
            "rope_delta":   round(m(total_rope_rL) - m(total_base_rL), 4),
            "std_delta":    round(m(total_std_rL)  - m(total_base_rL), 4),
            "rope_vs_std":  round(m(total_rope_rL) - m(total_std_rL),  4),
            "avg_mem_std_mb":  m(total_mem_std),
            "avg_mem_rope_mb": m(total_mem_rope),
        }, f, indent=2)

    return True

# ════════════════════════════════════════════════════════════
# MAIN SWEEP LOOP
# ════════════════════════════════════════════════════════════

def run_sweep(subsets=None, max_per_subset=150):
    """
    Iterate through PARAM_GRID from least to most memory.
    On OOM: log it, stop that config, move to next.
    Writes a master log at the end.
    """
    if subsets is None:
        subsets = ALL_SUBSETS

    master_log = os.path.join(LOG_DIR, "master_sweep_log.txt")
    log_line(master_log, f"=== SWEEP START {time.strftime('%Y-%m-%d %H:%M:%S')} ===")
    log_line(master_log, f"Subsets: {subsets}")
    log_line(master_log, f"Max per subset: {max_per_subset}")
    log_line(master_log, f"Grid size: {len(PARAM_GRID)} configs")

    completed = []
    oom_hit   = []

    for cfg_idx, (ctx, win, energy, re, rm, rl) in enumerate(PARAM_GRID):
        print(f"\n{'='*65}")
        print(f"SWEEP: Config {cfg_idx+1}/{len(PARAM_GRID)}")
        print(f"  ctx={ctx}  window={win}  energy={energy}  ranks={re}/{rm}/{rl}")
        print(f"  VRAM before config: {vram_used_mb():.0f} MB")
        print(f"{'='*65}")

        gc.collect(); torch.cuda.empty_cache()

        try:
            success = run_one_config(
                cfg_idx, ctx, win, energy, re, rm, rl,
                subsets, max_per_subset
            )
        except torch.cuda.OutOfMemoryError:
            success = False
            print(f"  [HARD OOM] Config {cfg_idx} hit OOM outside inner loop")
        except Exception as e:
            success = False
            print(f"  [ERROR] Config {cfg_idx}: {e}")
            traceback.print_exc()

        if success:
            completed.append(cfg_idx)
            log_line(master_log, f"Config {cfg_idx} COMPLETED (ctx={ctx} win={win})")
        else:
            oom_hit.append(cfg_idx)
            log_line(master_log, f"Config {cfg_idx} OOM/ERROR (ctx={ctx} win={win})")
            # Don't stop — continue to next config
            # OOM configs that are LARGER than this one will also fail,
            # but smaller configs already ran. We still try all.

        gc.collect(); torch.cuda.empty_cache()

    final = (f"\n=== SWEEP COMPLETE ===\n"
             f"Completed configs: {completed}\n"
             f"OOM/Error configs: {oom_hit}\n"
             f"Log dir: {LOG_DIR}\n"
             f"Finished: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(final)
    log_line(master_log, final)

# ════════════════════════════════════════════════════════════
# ENTRY POINT
# ─────────────────────────────────────────────────────────
# To run everything: run_sweep()
# To run just one subset quickly: run_sweep(["multifieldqa_en"], max_per_subset=30)
# To run specific configs only: modify PARAM_GRID before calling
# ════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # Start with just the English subsets that have answers field
    # Comment out subsets you don't have downloaded
    available = [s for s in ALL_SUBSETS
                 if os.path.exists(os.path.join(LONGBENCH_DATA_DIR, f"{s}.jsonl"))]
    print(f"Available subsets: {available}")

    run_sweep(subsets=available, max_per_subset=150)

Loading model...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...


SWEEP: Config 11/12
  ctx=4000  window=32  energy=0.85  ranks=96/80/64
  VRAM before config: 5309 MB
=== Config 10 | ctx=4000 window=32 energy=0.85 ranks=96/80/64 ===

--- Subset: multifieldqa_en (150 samples) ---
  [ERROR] Config 10: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!


Traceback (most recent call last):
  File "/tmp/ipykernel_24/746309360.py", line 577, in run_sweep
    success = run_one_config(
              ^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_24/746309360.py", line 441, in run_one_config
    out_r = run_lorc(tokens, cfn_rope, window)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_24/746309360.py", line 278, in run_lorc
    cache = compress_fn(out.past_key_values)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_24/746309360.py", line 360, in cfn_rope
    return compress_rope_aware(cache, window, energy, r_early, r_mid, r_late)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_24/746309360.py", line 202, in compress_rope_aware
    pre    = undo_rope(hk.float(), cos, sin, 0).to(hk.dtype)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_24/746309360.py", line 120, in undo_rope
    return (x * c) - (rotate_half(x) * si)
            ~~^~


SWEEP: Config 12/12
  ctx=4000  window=32  energy=0.8  ranks=80/64/48
  VRAM before config: 5309 MB
=== Config 11 | ctx=4000 window=32 energy=0.8 ranks=80/64/48 ===

--- Subset: multifieldqa_en (150 samples) ---
  [ERROR] Config 11: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!


Traceback (most recent call last):
  File "/tmp/ipykernel_24/746309360.py", line 577, in run_sweep
    success = run_one_config(
              ^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_24/746309360.py", line 441, in run_one_config
    out_r = run_lorc(tokens, cfn_rope, window)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_24/746309360.py", line 278, in run_lorc
    cache = compress_fn(out.past_key_values)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_24/746309360.py", line 360, in cfn_rope
    return compress_rope_aware(cache, window, energy, r_early, r_mid, r_late)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_24/746309360.py", line 202, in compress_rope_aware
    pre    = undo_rope(hk.float(), cos, sin, 0).to(hk.dtype)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_24/746309360.py", line 120, in undo_rope
    return (x * c) - (rotate_half(x) * si)
            ~~^~


=== SWEEP COMPLETE ===
Completed configs: []
OOM/Error configs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Log dir: /kaggle/working/lorc_sweep
Finished: 2026-04-22 14:29:25


In [3]:
import shutil
# Zip the entire log directory so it's easy to download from Kaggle Outputs
shutil.make_archive('/kaggle/working/lorc_sweep_results', 'zip', '/kaggle/working/lorc_sweep')
print("Sweep zipped and ready for download!")

Sweep zipped and ready for download!
